In [1]:
import pandas as pd
import seaborn as sns

taxis = sns.load_dataset("taxis")
taxis["pickup"] = pd.to_datetime(taxis["pickup"])
taxis["dropoff"] = pd.to_datetime(taxis["dropoff"])

print(taxis.shape)
print(taxis["pickup"].min())
print(taxis["pickup"].max())

(6433, 14)
2019-02-28 23:29:03
2019-03-31 23:43:45


In [2]:
par_jour = (
    taxis.set_index("pickup")
    .resample("D")["total"]
    .agg(["size", "sum"])
    .rename(columns={"size": "courses", "sum": "chiffre_affaires"})
    .round(2)
)
par_jour.head()

,courses,chiffre_affaires
pickup,,
2019-02-28,1,6.30
2019-03-01,241,4213.83
2019-03-02,198,3319.02
2019-03-03,169,3027.32
2019-03-04,171,3269.08


In [3]:
taxis["heure"] = taxis["pickup"].dt.hour
courses_par_heure = taxis["heure"].value_counts().sort_index()
heure_la_plus_chargee = courses_par_heure.idxmax()

print(f"Heure la plus chargée : {heure_la_plus_chargee} h ({courses_par_heure.max()} courses)")
courses_par_heure.sort_values(ascending=False).head(5)

Heure la plus chargée : 18 h (417 courses)


heure
18    417
19    406
17    388
20    368
14    360
Name: count, dtype: int64

In [4]:
taxis["jour_semaine"] = taxis["pickup"].dt.day_name()
pourboire_moyen_par_jour = (
    taxis.groupby("jour_semaine")["tip"].mean().round(2).sort_values(ascending=False)
)
pourboire_moyen_par_jour

jour_semaine
Wednesday    2.09
Thursday     2.06
Monday       2.03
Tuesday      2.00
Friday       1.98
Sunday       1.91
Saturday     1.82
Name: tip, dtype: float64

In [5]:
taxis["duree_min"] = (taxis["dropoff"] - taxis["pickup"]).dt.total_seconds() / 60
taxis["duree_min"].describe().round(1)

count    6433.0
mean       14.3
std        11.6
min         0.0
25%         6.5
50%        10.9
75%        18.5
max       107.7
Name: duree_min, dtype: float64

In [6]:
durees_nulles = taxis[taxis["duree_min"] <= 0]

courses_avec_duree = taxis[taxis["duree_min"] > 0]
vitesse_mph = courses_avec_duree["distance"] / (courses_avec_duree["duree_min"] / 60)
vitesses_impossibles = courses_avec_duree[vitesse_mph > 80]

print(f"Durées nulles ou négatives : {len(durees_nulles)}")
print(f"Vitesses supérieures à 80 mph : {len(vitesses_impossibles)}")
print(f"Durée maximale : {taxis['duree_min'].max():.1f} min")

Durées nulles ou négatives : 6
Vitesses supérieures à 80 mph : 11
Durée maximale : 107.7 min


In [7]:
index_aberrants = durees_nulles.index.union(vitesses_impossibles.index)
taxis_propres = taxis.drop(index=index_aberrants)
print(f"Lignes avant : {len(taxis)}, après : {len(taxis_propres)}")

Lignes avant : 6433, après : 6416


## Réponses

- **Période** : du 28 février 2019 (une seule course, en fin de soirée) au 31 mars 2019. Le mois étudié est mars 2019.
- **Heure la plus chargée** : 18 h, avec 417 courses.
- **Jour au pourboire moyen le plus élevé** : le mercredi (2,09), devant le jeudi (2,06). Le samedi est le plus bas (1,82).

## Durées aberrantes : décision
- **Durées négatives ou de plusieurs heures** : aucune. La durée maximale est de 107,7 min, pour des trajets de plus de 20 miles, donc plausible. Ces longues courses sont conservées.
- **Durées nulles** : 6 courses où le départ et l'arrivée ont la même heure. Elles sont supprimées.
- **Vitesses impossibles** : 11 courses dont la distance divisée par la durée dépasse 80 mph (par exemple 9,4 miles en 7 secondes). Elles sont supprimées.
- **Lignes touchées** : 17 supprimées, il reste 6 416 courses dans `taxis_propres`.